In [ ]:
from huggingface_hub import login
import os

# Haal token uit Colab secrets
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    print("HF token gevonden, inloggen...")
    login(token=hf_token)
else:
    raise ValueError("❌ Geen HF_TOKEN gevonden! Voeg deze toe in Colab secrets.")

In [ ]:
import sys, os, pathlib, re
import torch

print("CUDA available:", torch.cuda.is_available())

# Install dependencies
!pip install -q ultralytics albumentations decord

# Clone SAM3
repo_dir = "/content/sam3"

if os.path.exists(repo_dir):
    !rm -rf {repo_dir}

!git clone -q https://github.com/facebookresearch/sam3.git {repo_dir}

# Fix numpy conflict
pyproject = pathlib.Path(repo_dir) / "pyproject.toml"
text = pyproject.read_text()
text = re.sub(r'"numpy==[^"]+",?', "", text)
pyproject.write_text(text)

# Install SAM3
%cd {repo_dir}
!pip install -q -e .
%cd /content

# Add to path
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive"
ZIP_NAME = "duckietown_dataset.zip"
DATASET_PATH = "/content/duckietown_dataset"

zip_path = os.path.join(DRIVE_PATH, ZIP_NAME)
assert os.path.exists(zip_path), "Dataset zip niet gevonden!"

print("Unzipping dataset...")
shutil.unpack_archive(zip_path, DATASET_PATH)

print("Dataset klaar!")

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/duckietown_dataset")
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
CLASSES_YAML = DATASET_DIR / "classes.yaml"

# classes.yaml maken
with open(CLASSES_YAML, "w") as f:
    f.write("""
train: /content/duckietown_dataset/train
val:   /content/duckietown_dataset/val

names:
  0: 'yellow rubber duck'
  1: 'red traffic light'
  2: 'green traffic light'
""")

print("classes.yaml aangemaakt")

In [ ]:
import yaml
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

def load_classes(path):
    with open(path) as f:
        return [v for k, v in sorted(yaml.safe_load(f)["names"].items())]

def xyxy_to_yolo(box, w, h):
    x1, y1, x2, y2 = box
    return [(x1+x2)/2/w, (y1+y2)/2/h, (x2-x1)/w, (y2-y1)/h]

class Sam3AutoLabel:
    def __init__(self):
        import torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.classes = load_classes(CLASSES_YAML)

        self.model = build_sam3_image_model(
            bpe_path="/content/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
        ).to(self.device).eval()

        self.processor = Sam3Processor(self.model, device=self.device)

    def process_split(self, split_dir):
        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"
        lbl_dir.mkdir(exist_ok=True)

        for img_path in tqdm(list(img_dir.glob("*"))):
            image = Image.open(img_path).convert("RGB")
            w, h = image.size

            state = self.processor.set_image(image)
            lines = []

            for class_id, prompt in enumerate(self.classes):
                out = self.processor.set_text_prompt(state=state, prompt=prompt)

                if out.get("boxes") is None:
                    continue

                for box in out["boxes"]:
                    cx, cy, bw, bh = xyxy_to_yolo(box.tolist(), w, h)
                    lines.append(f"{class_id} {cx} {cy} {bw} {bh}")

            if lines:
                (lbl_dir / f"{img_path.stem}.txt").write_text("\n".join(lines))

    def run(self):
        self.process_split(TRAIN_DIR)
        self.process_split(VAL_DIR)

Sam3AutoLabel().run()

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = list((TRAIN_DIR/"images").glob("*"))[0]
lbl = TRAIN_DIR/"labels"/(img.stem + ".txt")

img_cv = cv2.imread(str(img))
h, w = img_cv.shape[:2]

if lbl.exists():
    with open(lbl) as f:
        for line in f:
            cls, cx, cy, bw, bh = map(float, line.split())
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)
            cv2.rectangle(img_cv, (x1,y1), (x2,y2), (0,255,0), 2)

plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = list((TRAIN_DIR/"images").glob("*"))[0]
lbl = TRAIN_DIR/"labels"/(img.stem + ".txt")

img_cv = cv2.imread(str(img))
h, w = img_cv.shape[:2]

if lbl.exists():
    with open(lbl) as f:
        for line in f:
            cls, cx, cy, bw, bh = map(float, line.split())
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)
            cv2.rectangle(img_cv, (x1,y1), (x2,y2), (0,255,0), 2)

plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import albumentations as A
import cv2
from tqdm import tqdm

IMG_DIR = TRAIN_DIR / "images"
LBL_DIR = TRAIN_DIR / "labels"

OUT_IMG = DATASET_DIR / "train_aug/images"
OUT_LBL = DATASET_DIR / "train_aug/labels"

OUT_IMG.mkdir(parents=True, exist_ok=True)
OUT_LBL.mkdir(parents=True, exist_ok=True)

transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.5, 0.9), p=0.6),
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.3
    )
)

def load_labels(path):
    boxes, labels = [], []
    with open(path) as f:
        for line in f:
            cls, cx, cy, bw, bh = map(float, line.split())
            boxes.append([cx, cy, bw, bh])
            labels.append(int(cls))
    return boxes, labels

def save_labels(path, boxes, labels):
    with open(path, "w") as f:
        for b, l in zip(boxes, labels):
            f.write(f"{l} {' '.join(map(str,b))}\n")

for img_path in tqdm(list(IMG_DIR.glob("*"))):
    lbl_path = LBL_DIR / (img_path.stem + ".txt")

    if not lbl_path.exists():
        continue

    img = cv2.imread(str(img_path))
    boxes, labels = load_labels(lbl_path)

    for i in range(3):
        aug = transform(image=img, bboxes=boxes, class_labels=labels)

        new_img = OUT_IMG / f"aug_{i}_{img_path.name}"
        new_lbl = OUT_LBL / f"aug_{i}_{img_path.stem}.txt"

        cv2.imwrite(str(new_img), aug["image"])
        save_labels(new_lbl, aug["bboxes"], aug["class_labels"])

In [ ]:
import shutil

for f in OUT_IMG.glob("*"):
    shutil.copy(f, TRAIN_DIR/"images"/f.name)

for f in OUT_LBL.glob("*"):
    shutil.copy(f, TRAIN_DIR/"labels"/f.name)

print("Augmented data toegevoegd")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.yaml")

model.train(
    data=str(CLASSES_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
)

In [ ]:
model.export(format="onnx")

print("ONNX model klaar!")